# Índices Climáticos Extremos y Estrés Agrícola

## Incorporación de SPI, SPEI, olas de calor y variables de estrés agroclimático

En este notebook se incorporan índices climáticos avanzados para mejorar la representación de sequías, excesos hídricos, olas de calor y estrés atmosférico sobre el rendimiento agrícola.

A partir del modelo agroclimático avanzado construido previamente, se incorporarán nuevas variables como SPI, SPEI, olas de calor, extremos térmicos y otros indicadores de estrés climático, con el objetivo de mejorar la capacidad predictiva y la interpretación agroclimática del sistema.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
dataset = pd.read_csv(
    "../Procesados/dataset_agroclimatico_avanzado.csv"
)

dataset.head()

,Cultivo,Campaña,Provincia,Departamento,idProvincia,idDepartamento,Rendimiento,temp_media_prom,temp_max_extrema,temp_min_extrema,...,depto_key,score_suelo_promedio,humedad_relativa_prom,humedad_relativa_min,humedad_relativa_max,ENSO,ENSO_num,VPD_prom,VPD_max,VPD_min
0,Soja total,1980/81,SANTA FE,CASEROS,82,14,2156,17.415370,39.00,-4.00,...,CASEROS,2.516667,75.053465,42.396712,98.538970,Neutral,0,0.537907,1.663234,0.018074
1,Soja total,1981/82,SANTA FE,CASEROS,82,14,2500,18.083746,38.00,-1.22,...,CASEROS,2.516667,71.674062,45.077907,98.262247,Neutral,0,0.630672,1.838525,0.037406
2,Soja total,1982/83,SANTA FE,CASEROS,82,14,1386,16.964637,38.22,-4.72,...,CASEROS,2.516667,70.703671,38.703611,100.000000,Niño,1,0.640450,2.376901,0.000000
3,Soja total,1983/84,SANTA FE,CASEROS,82,14,2574,17.223060,38.61,-4.89,...,CASEROS,2.516667,74.032805,41.458937,98.594646,Neutral,0,0.569227,2.490169,0.024136
4,Soja total,1987/88,SANTA FE,CASEROS,82,14,2438,16.424200,36.00,-6.50,...,CASEROS,2.516667,74.056099,35.443643,100.000000,Neutral,0,0.521765,2.143217,0.000000


In [3]:
dataset.columns

Index(['Cultivo', 'Campaña', 'Provincia', 'Departamento', 'idProvincia',
       'idDepartamento', 'Rendimiento', 'temp_media_prom', 'temp_max_extrema',
       'temp_min_extrema', 'precipitacion_total', 'viento_prom',
       'presion_prom', 'dias_lluvia', 'depto_key', 'score_suelo_promedio',
       'humedad_relativa_prom', 'humedad_relativa_min', 'humedad_relativa_max',
       'ENSO', 'ENSO_num', 'VPD_prom', 'VPD_max', 'VPD_min'],
      dtype='str')

In [4]:
dataset["SPI_precipitacion"] = (
    dataset["precipitacion_total"] - dataset["precipitacion_total"].mean()
) / dataset["precipitacion_total"].std()

dataset[
    [
        "Campaña",
        "precipitacion_total",
        "SPI_precipitacion"
    ]
].head()

,Campaña,precipitacion_total,SPI_precipitacion
0,1980/81,1241.298,0.823807
1,1981/82,534.924,-1.550292
2,1982/83,663.956,-1.116620
3,1983/84,1160.780,0.553188
4,1987/88,815.848,-0.606116


In [5]:
def clasificar_spi(spi):
    if spi <= -1.5:
        return "Sequía severa"
    elif spi <= -1.0:
        return "Sequía moderada"
    elif spi >= 1.5:
        return "Muy húmeda"
    elif spi >= 1.0:
        return "Húmeda"
    else:
        return "Normal"

dataset["categoria_SPI"] = dataset["SPI_precipitacion"].apply(clasificar_spi)

dataset[
    [
        "Campaña",
        "precipitacion_total",
        "SPI_precipitacion",
        "categoria_SPI"
    ]
].head(15)

,Campaña,precipitacion_total,SPI_precipitacion,categoria_SPI
0,1980/81,1241.298,0.823807,Normal
1,1981/82,534.924,-1.550292,Sequía severa
2,1982/83,663.956,-1.116620,Sequía moderada
3,1983/84,1160.780,0.553188,Normal
4,1987/88,815.848,-0.606116,Normal
5,1988/89,464.058,-1.788470,Sequía severa
6,1989/90,971.804,-0.081953,Normal
7,1990/91,1987.550,3.331934,Muy húmeda
8,1991/92,1013.460,0.058051,Normal
9,1992/93,980.440,-0.052928,Normal


In [6]:
dataset["categoria_SPI"].value_counts()

categoria_SPI
Normal             186
Sequía severa       18
Sequía moderada     18
Muy húmeda          12
Húmeda               6
Name: count, dtype: int64

In [7]:
dataset.groupby("categoria_SPI")["Rendimiento"].mean().sort_values()

categoria_SPI
Sequía severa      1734.555556
Sequía moderada    2021.888889
Húmeda             2936.000000
Normal             2938.586022
Muy húmeda         3014.250000
Name: Rendimiento, dtype: float64

# Incorporación de SPI al Análisis Agroclimático

## SPI — Standardized Precipitation Index

Con el objetivo de representar anomalías climáticas de precipitación y caracterizar campañas secas o húmedas, se incorporó SPI (Standardized Precipitation Index) al sistema agroclimático.

SPI constituye uno de los índices climáticos más utilizados en climatología y monitoreo de sequías, ya que permite cuantificar cuánto se desvía una campaña respecto al comportamiento histórico normal de precipitaciones.

A diferencia de utilizar precipitación absoluta, SPI permite interpretar los eventos climáticos en términos relativos a la climatología histórica del sistema analizado.

---

# Metodología

SPI fue calculado mediante estandarización de la precipitación total histórica de cada campaña agrícola:

* valores positivos indican campañas más húmedas que lo normal,
* valores negativos indican campañas más secas que lo normal.

Posteriormente las campañas fueron clasificadas en categorías climáticas:

* Muy húmeda,
* Húmeda,
* Normal,
* Sequía moderada,
* y Sequía severa.

---

# Resultados obtenidos

La clasificación climática mostró:

| Categoría       | Cantidad de campañas |
| --------------- | -------------------- |
| Normal          | 186                  |
| Sequía severa   | 18                   |
| Sequía moderada | 18                   |
| Muy húmeda      | 12                   |
| Húmeda          | 6                    |

---

# Relación entre SPI y rendimiento agrícola

El análisis mostró diferencias muy importantes en rendimiento promedio según categoría climática.

| Categoría SPI   | Rendimiento promedio aproximado |
| --------------- | ------------------------------- |
| Sequía severa   | ~1735                           |
| Sequía moderada | ~2022                           |
| Normal          | ~2939                           |
| Muy húmeda      | ~3014                           |

Los resultados muestran una fuerte reducción del rendimiento durante campañas clasificadas como sequía severa y sequía moderada.

---

# Interpretación agroclimática

Los resultados obtenidos indican que las anomalías de precipitación tienen un impacto directo y significativo sobre el rendimiento agrícola.

SPI permitió representar de forma más precisa:

* intensidad de sequías,
* anomalías hídricas,
* y comportamiento relativo de las campañas agrícolas respecto al clima histórico.

Además, los resultados muestran que el uso de índices climáticos estandarizados permite interpretar de forma más robusta el impacto climático sobre la productividad agrícola en comparación con el uso de precipitación absoluta únicamente.

---

# Conclusión preliminar

La incorporación de SPI permitió mejorar la caracterización climática de las campañas agrícolas y evidenciar el fuerte impacto de las sequías sobre el rendimiento de soja.

Los resultados obtenidos refuerzan la importancia del componente hídrico dentro del sistema agroclimático y constituyen una base sólida para la incorporación posterior de índices más avanzados como SPEI.


In [8]:
dataset["balance_hidrico_atm"] = (
    dataset["precipitacion_total"] /
    (dataset["VPD_prom"] + 0.01)
)

In [9]:
dataset["SPEI_simple"] = (
    dataset["balance_hidrico_atm"] -
    dataset["balance_hidrico_atm"].mean()
) / dataset["balance_hidrico_atm"].std()

In [10]:
dataset[
    [
        "Campaña",
        "balance_hidrico_atm",
        "SPEI_simple"
    ]
].head()

,Campaña,balance_hidrico_atm,SPEI_simple
0,1980/81,2265.525099,1.305463
1,1981/82,834.941963,-1.183627
2,1982/83,1020.764829,-0.860312
3,1983/84,2004.015366,0.850458
4,1987/88,1534.227587,0.033069


In [11]:
def clasificar_spei(valor):

    if valor <= -1.5:
        return "Estrés severo"

    elif valor <= -1:
        return "Estrés moderado"

    elif valor >= 1.5:
        return "Muy favorable"

    elif valor >= 1:
        return "Favorable"

    else:
        return "Normal"

In [12]:
dataset["categoria_SPEI"] = (
    dataset["SPEI_simple"]
    .apply(clasificar_spei)
)

In [13]:
dataset[
    [
        "Campaña",
        "SPEI_simple",
        "categoria_SPEI"
    ]
].head(15)

,Campaña,SPEI_simple,categoria_SPEI
0,1980/81,1.305463,Favorable
1,1981/82,-1.183627,Estrés moderado
2,1982/83,-0.860312,Normal
3,1983/84,0.850458,Normal
4,1987/88,0.033069,Normal
5,1988/89,-1.485906,Estrés moderado
6,1989/90,0.267196,Normal
7,1990/91,3.534739,Muy favorable
8,1991/92,0.558046,Normal
9,1992/93,0.644989,Normal


In [14]:
dataset["categoria_SPEI"].value_counts()

categoria_SPEI
Normal             180
Estrés moderado     36
Muy favorable       12
Favorable            6
Estrés severo        6
Name: count, dtype: int64

In [15]:
dataset.groupby(
    "categoria_SPEI"
)["Rendimiento"].mean().sort_values()

categoria_SPEI
Estrés severo      1368.833333
Favorable          2052.666667
Estrés moderado    2334.888889
Normal             2929.022222
Muy favorable      3014.250000
Name: Rendimiento, dtype: float64

# Incorporación de SPEI al Sistema Agroclimático

## SPEI — Standardized Precipitation Evapotranspiration Index

Con el objetivo de representar de forma más realista el balance hídrico-atmosférico del sistema agrícola, se incorporó un índice simplificado tipo SPEI (Standardized Precipitation Evapotranspiration Index) al análisis agroclimático.

A diferencia de SPI, que considera únicamente la precipitación, SPEI incorpora simultáneamente:

* disponibilidad hídrica,
* y demanda evaporativa atmosférica.

Esto permite representar de forma más precisa las condiciones reales de estrés hídrico que afectan al cultivo.

---

# Fundamentación agroclimática

En sistemas agrícolas reales, el rendimiento no depende exclusivamente de cuánto llueve, sino también de:

* temperatura,
* humedad atmosférica,
* viento,
* evaporación,
* evapotranspiración,
* y demanda hídrica de la atmósfera.

Una campaña puede presentar precipitaciones normales pero igualmente sufrir estrés hídrico si las condiciones atmosféricas generan una demanda evaporativa elevada.

Por este motivo, SPEI constituye uno de los índices más utilizados en estudios de sequía agrícola y cambio climático.

---

# Construcción del índice

Debido a la ausencia de datos completos de evapotranspiración potencial, se construyó una versión simplificada de SPEI utilizando:

* precipitación total,
* y VPD (Déficit de Presión de Vapor).

Se generó un balance hídrico-atmosférico simplificado:

* valores altos representan campañas con buena disponibilidad hídrica y bajo estrés atmosférico,
* valores bajos representan campañas con escasez hídrica y alta demanda evaporativa.

Posteriormente, los valores fueron estandarizados para generar el índice SPEI simplificado.

---

# Clasificación climática obtenida

Las campañas fueron clasificadas en:

* Muy favorable,
* Favorable,
* Normal,
* Estrés moderado,
* y Estrés severo.

Distribución observada:

| Categoría       | Cantidad de campañas |
| --------------- | -------------------- |
| Normal          | 180                  |
| Estrés moderado | 36                   |
| Muy favorable   | 12                   |
| Favorable       | 6                    |
| Estrés severo   | 6                    |

---

# Relación entre SPEI y rendimiento agrícola

El análisis mostró diferencias extremadamente marcadas en rendimiento promedio según categoría SPEI.

| Categoría SPEI  | Rendimiento promedio aproximado |
| --------------- | ------------------------------- |
| Estrés severo   | ~1369                           |
| Estrés moderado | ~2335                           |
| Normal          | ~2929                           |
| Muy favorable   | ~3014                           |

Los resultados muestran una caída muy importante del rendimiento bajo condiciones de estrés hídrico-atmosférico severo.

---

# Comparación conceptual entre SPI y SPEI

SPI permitió identificar anomalías de precipitación y caracterizar campañas secas o húmedas.

Sin embargo, SPEI mostró una mayor capacidad para representar campañas realmente críticas para el cultivo, debido a que incorpora simultáneamente:

* disponibilidad hídrica,
* y demanda evaporativa atmosférica.

Los resultados obtenidos sugieren que las variables atmosféricas asociadas al estrés hídrico tienen un rol fundamental en la productividad agrícola.

---

# Interpretación agroclimática

Los resultados obtenidos refuerzan la hipótesis central del proyecto:

El rendimiento agrícola depende principalmente del equilibrio entre:

* oferta hídrica,
* demanda atmosférica,
* humedad relativa,
* y estrés evaporativo.

La incorporación de SPEI permitió representar parcialmente la interacción atmósfera-cultivo y capturar de forma más robusta los efectos de campañas climáticamente extremas.

---

# Conclusión preliminar

La incorporación de SPEI constituyó una mejora conceptual importante dentro del sistema agroclimático desarrollado.

El índice permitió representar de forma más realista el estrés hídrico agrícola y mostró una relación muy fuerte con el rendimiento histórico de soja.

Los resultados sugieren que el balance hídrico-atmosférico constituye uno de los principales determinantes del comportamiento productivo del sistema agrícola analizado.


In [17]:
dataset.to_csv(
    "../Procesados/dataset_agroclimatico_indices_extremos.csv",
    index=False
)

In [18]:
dataset[
    [
        "Campaña",
        "SPI_precipitacion",
        "categoria_SPI",
        "SPEI_simple",
        "categoria_SPEI"
    ]
].to_csv(
    "../Procesados/indices_climaticos_spi_spei.csv",
    index=False
)